# VideoDB Indexing V2: Indexing Guide

Index Understanding analyzer outputs, then use search, semantic search, structured query, and aggregation.

Indexing answers: **which understanding fields should become searchable, queryable, or aggregatable?**


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/indexing/indexing_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 1. Install dependencies

In [ ]:
!pip install -q videodb python-dotenv


## 2. Connect to VideoDB

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
print("Connected to VideoDB")


## 3. Choose a video

By default, this notebook uploads the sample video used in the E2E flow: **Silicon Valley - Gilfoyle is free for hire**. To use an existing video instead, comment the upload line and uncomment the `get_video` lines in the next cell.


In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

collection = conn.get_collection()
video = collection.upload(VIDEO_URL)

# To use an existing video instead, comment the upload line above and uncomment these lines:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


## 4. Prepare analyzer outputs

Indexing needs source outputs. Set `VIDEODB_UNDERSTANDING_ID` to reuse an existing run, or let this cell create a small Understanding run.


In [ ]:
OBJECT_LABELS = [
    "person",
    "cup",
    "bottle",
    "chair",
    "sofa",
    "diningtable",
    "laptop",
    "cell phone",
    "book",
]


In [ ]:
UNDERSTANDING_ID = os.getenv("VIDEODB_UNDERSTANDING_ID")

if UNDERSTANDING_ID:
    understanding = video.get_understanding(UNDERSTANDING_ID)
    print("Using existing Understanding:", understanding.id)
else:
    understanding = video.understand(
        analyzers=[
            {"type": "spoken_words", "name": "transcript", "config": {"language": "en"}},
            {
                "type": "object_detection",
                "name": "objects",
                "sampling": {"strategy": "interval", "every": 1},
                "config": {
                    "labels": OBJECT_LABELS,
                    "confidence_threshold": 0.35,
                    "include_bounding_boxes": True,
                },
            },
            {
                "type": "vlm",
                "name": "scene",
                "inputs": ["transcript", "objects"],
                "sampling": {"strategy": "uniform", "frame_count": 3},
                "config": {
                    "model": "ultra",
                    "prompt": (
                        "Using the sampled frames, detected objects, and transcript, "
                        "describe what happens in this video scene by scene. "
                        "Return the description in the outputs field."
                    ),
                    "schema": {"outputs": "text"},
                },
            },
        ],
        segmentation={"type": "shot", "threshold": 30},
    )
    print("Created Understanding:", understanding.id)

understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Final status:", understanding.status)


In [ ]:
def as_segments(output):
    """Handle both list outputs and {'scenes': [...]} outputs."""
    return output.get("scenes", output) if isinstance(output, dict) else output


def preview_segments(name, output, max_segments=3):
    segments = as_segments(output) or []
    print(f"{name}: {len(segments)} segments")
    print("=" * 60)
    for segment in segments[:max_segments]:
        print(f"{segment.get('start')}s → {segment.get('end')}s")
        print(segment.get("data"))
        print("-" * 60)


In [ ]:
transcript = understanding.get_analyzer("transcript").get_output()
objects = understanding.get_analyzer("objects").get_output()
scene = understanding.get_analyzer("scene").get_output()

preview_segments("transcript", transcript)
preview_segments("objects", objects)
preview_segments("scene", scene)


## 5. Create indexes

An **index** turns the fields of an understanding artifact into retrieval-ready storage. You control two things: what the index can be used for (`use_for`), and how each field is indexed (`fields`).

### `use_for` — retrieval capabilities

Declare what you intend to do with the index:

| Capability | Enables | Method |
|---|---|---|
| `semantic` | meaning / vector search | `semantic_search()`, `search()` |
| `query` | structured filters | `query()` |
| `aggregate` | counts, group-by, facets | `aggregate()` |

Omit `use_for` and it defaults to `["semantic", "query", "aggregate"]`. Drop `semantic` (e.g. `["query"]`) to skip embeddings — a query-only index is cheaper and ready instantly. The value `get_index()` reports back is the index's **effective** capability: `aggregate` only appears when a field is groupable, and an index with no rows reports `[]`.

### `fields` — how each field is indexed

`fields` maps **field groups** to the artifact fields that should join them. A field's group decides what it can do:

| Group | What the field can do |
|---|---|
| `semantic` | embedded for meaning search — "find talk **about** X" |
| `filter` | structured `query()` filters (exact match, `contains`, numeric ranges) |
| `aggregate` | `group_by` / counts / facets |
| `sort` | result ordering |

A field can belong to several groups. Names may be **dotted paths** into nested data — `frames.detections.label` reaches inside every detection.

### Defaults — you don't have to declare everything

Omit `fields` (or any single group) and VideoDB derives it from your data: well-known names get product defaults (`text` / `scene_description` → `semantic`; `language` / `brand_names` → `filter` + `aggregate`), and everything else is classified by shape (prose → `semantic`; scalars and scalar lists → `filter` + `aggregate`, numbers also `sort`; nested objects stay stored-only). An explicit empty group opts out — `{"filter": []}` means "no filter fields".

### The build is asynchronous (rows-first)

Every semantic index returns as `building` and flips to `ready` once its embeddings are written. Indexing is **rows-first**: structured records are stored before `index()` returns, so `query()` and `aggregate()` work on a `building` index immediately — only `semantic_search()` waits for `ready`. Block on a build with `index.wait_until_complete()`.

---

Below we build three indexes:

- **Transcript** — semantic + query (spoken words)
- **Objects** — query + aggregate (detected labels / scores)
- **Scene VLM** — semantic + query over the `outputs` description


In [ ]:
from datetime import datetime

run_suffix = datetime.utcnow().strftime("%Y%m%d%H%M%S")

transcript_index_name = f"transcript_{run_suffix}"
objects_index_name = f"objects_{run_suffix}"
scene_index_name = f"scene_{run_suffix}"

transcript_index = video.index(
    name=transcript_index_name,
    source=transcript,
    use_for=["semantic", "query"],
)

objects_index = video.index(
    name=objects_index_name,
    source=objects,
    use_for=["query", "aggregate"],
    fields={
        "filter": ["frames.detections.label", "frames.detections.score"],
        "aggregate": ["frames.detections.label"],
        "sort": ["frames.detections.score"],
    },
)

scene_index = video.index(
    name=scene_index_name,
    source=scene,
    use_for=["semantic", "query"],
    fields={
        "semantic": ["outputs"],
        "filter": ["outputs"],
    },
)

transcript_index, objects_index, scene_index


## 6. Inspect indexes

In [ ]:
# Index builds are asynchronous — wait with the SDK's built-in helper.
# Query-only indexes (objects) are ready at once; semantic indexes (scene, transcript)
# flip from `building` to `ready` once their embeddings are written.
for index in (transcript_index, objects_index, scene_index):
    index.wait_until_complete(timeout=900, poll_interval=10)
    print(index.name, "->", index.status)
    if not index.is_successful:
        print("   build failed:", index.error)


In [ ]:
print("Scene index fields:", scene_index.fields)

records = objects_index.records(limit=10)
records[:1]


## Choosing a retrieval method

You now have three indexes. Each retrieval method answers a different question:

| Method | Answers | How |
|---|---|---|
| `video.search(query=...)` | "find moments about X" in natural language — VideoDB plans across indexes | semantic + filters |
| `video.semantic_search(query=..., index_names=[...])` | "find talk **about** X" (meaning) on chosen semantic indexes | vector similarity |
| `video.query(index_name=..., filter=...)` | exact / structured lookup on one index's fields | field filters |
| `video.aggregate(index_name=..., group_by=...)` | counts, facets, "how many X" | group-by |

A field must be in the `filter` group to appear in a `query()` filter, and in `aggregate` to be a `group_by` — inspect the mapping with `index.field_schema`. `query()`/`aggregate()` target a **single** index; `semantic_search()` can span several via `index_names`.


## 7. Search one video

In [ ]:
results = video.search(
    query="gilfoyle throwing coffee in dustbin",
    top_k=5,
    mode="default",
    return_fields="all",
)

results


In [ ]:
if results:
    first = results[0]
    print(first.start, first.end)
    print(first.metadata)
    first.play()
else:
    print("No results found. Try a different query for your video.")


## 8. Direct semantic search

In [ ]:
semantic_results = video.semantic_search(
    query="gilfoyle throwing coffee in dustbin",
    index_names=[scene_index_name],
    top_k=5,
    return_fields="all",
)

semantic_results


## 9. Structured query

In [ ]:
object_results = video.query(
    index_name=objects_index_name,
    filter=[{"field": "frames.detections.label", "op": "contains", "value": "person"}],
    limit=10,
)

object_results


In [ ]:
scene_query_results = video.query(
    index_name=scene_index_name,
    filter=[{"field": "outputs", "op": "contains", "value": "coffee"}],
    limit=10,
)

scene_query_results


## 10. Aggregation

In [ ]:
object_counts = video.aggregate(
    index_name=objects_index_name,
    group_by="frames.detections.label",
    metric="count",
    limit=20,
)

object_counts


## 11. Collection search

In [ ]:
collection_results = collection.search(
    query="gilfoyle throwing coffee in dustbin",
    top_k=5,
    mode="default",
    return_fields="all",
)

collection_results


## 12. Compile search results into a stream

In [ ]:
if results:
    stream_url = results.results.compile()
    stream_url
else:
    print("No search results to compile.")


## 13. Optional cleanup

In [ ]:
DELETE_INDEXES = False

if DELETE_INDEXES:
    video.delete_index(index_id=transcript_index.index_id)
    video.delete_index(index_id=objects_index.index_id)
    video.delete_index(index_id=scene_index.index_id)
    print("Deleted indexes")
else:
    print("Skipping delete. Set DELETE_INDEXES=True to delete these indexes.")


In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Skipping delete. Set DELETE_UNDERSTANDING=True to delete this Understanding.")


## Method map

```text
Index:
video.index(...)
video.list_indexes()
video.get_index(...)
index.records(...)
video.delete_index(...)

Search:
video.search(...)
video.semantic_search(...)
video.query(...)
video.aggregate(...)
collection.search(...)
results.compile()
```
